# Restriction site scan

讀取 `../tables/assembled_scan.csv`，對每條 **full `Sequence`** 掃描下列限制酶切位。

- 同時掃正股與反股（非回文的 **Eco31I (GGTCTC)** 才需要，回文酶兩股相同）。
- 注意：BG5 / BG3 兩端背景是固定的，落在背景內的切位會出現在**每一條**序列。

In [ ]:
# === Path bootstrap (shared by 01-07) ===
# Locates MS2_Data_PyTorch/scripts/library_release by walking up from the cwd,
# then imports _paths, which sets every other path absolutely and puts
# MS2_Data_PyTorch/scripts on sys.path. Safe to run from any working directory.
import sys
from pathlib import Path

for _c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _rel = _c / "MS2_Data_PyTorch" / "scripts" / "library_release"
    if (_rel / "_paths.py").exists():
        if str(_rel) not in sys.path:
            sys.path.insert(0, str(_rel))
        break
else:
    raise RuntimeError(f"library_release not found from {Path.cwd()}")

from _paths import *  # noqa: F401,F403

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("RELEASE_OUT  :", RELEASE_OUT)


In [ ]:
from pathlib import Path
import pandas as pd

# 限制酶辨識位點 (5'->3')
RE_SITES = {
    'BamHI':   'GGATCC',
    'BcuI':    'ACTAGT',   # = SpeI
    'BglII':   'AGATCT',
    'Eco31I':  'GGTCTC',   # = BsaI, 非回文 -> 連反股 GAGACC 一起掃
    'EcoRI':   'GAATTC',
    'HindIII': 'AAGCTT',
    'KpnI':    'GGTACC',
    'MluI':    'ACGCGT',
    'NcoI':    'CCATGG',
    'NdeI':    'CATATG',
    'NheI':    'GCTAGC',
    'NotI':    'GCGGCCGC',
    'PstI':    'CTGCAG',
    'SacI':    'GAGCTC',
    'SalI':    'GTCGAC',
    'SmaI':    'CCCGGG',
    'VspI':    'ATTAAT',   # = AseI
    'XbaI':    'TCTAGA',
    'XhoI':    'CTCGAG',
}

_comp = str.maketrans('ACGT', 'TGCA')
def revcomp(s):
    return s.translate(_comp)[::-1]

def find_sites(seq, site):
    '''回傳 site 在 seq 的所有起始位置 (正反股, 0-indexed)。'''
    seq = seq.upper()
    positions = []
    for p in {site, revcomp(site)}:
        start = 0
        while True:
            i = seq.find(p, start)
            if i == -1:
                break
            positions.append(i)
            start = i + 1
    return sorted(set(positions))

In [ ]:
# Add CSV/XLSX files here.
# Reads the assembled library from 06. The previous default,
# tables/assembled_scan.csv, is a 93 nt artefact from an older BG5/BG3 layout
# and no longer describes anything that gets ordered.
INPUT_FILES = [
    require(RELEASE_OUT / '06_whole_sequence.csv', 'assembled library from 06'),
]

SEQUENCE_COL = 'full_sequence'
OUTPUT_DIR = RELEASE_OUT / 're_scan_detail'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def read_input_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    raise ValueError(f'Unsupported file type: {path}')

In [ ]:
def scan_one_file(path):
    path = Path(path)
    df = read_input_table(path)

    if SEQUENCE_COL not in df.columns:
        raise ValueError(f'{path} missing column: {SEQUENCE_COL}')

    enz_cols = list(RE_SITES.keys())
    res = df.copy()
    for enz, site in RE_SITES.items():
        res[enz] = [len(find_sites(s, site)) for s in df[SEQUENCE_COL].astype(str)]

    res['total_sites'] = res[enz_cols].sum(axis=1)
    res['clean'] = res['total_sites'] == 0
    res['source_file'] = path.name

    summary = pd.DataFrame({
        'source_file': path.name,
        'enzyme': enz_cols,
        'site': [RE_SITES[e] for e in enz_cols],
        'n_seqs_with_site': [(res[e] > 0).sum() for e in enz_cols],
        'total_occurrences': [res[e].sum() for e in enz_cols],
    }).sort_values('n_seqs_with_site', ascending=False)

    absent = summary[summary['total_occurrences'] == 0].copy()

    res.to_csv(OUTPUT_DIR / f'{path.stem}_RE_scan.csv', index=False)
    summary.to_csv(OUTPUT_DIR / f'{path.stem}_RE_summary.csv', index=False)
    absent.to_csv(OUTPUT_DIR / f'{path.stem}_absent_RE_sites.csv', index=False)

    clean_n = int(res['clean'].sum())
    clean_pct = res['clean'].mean()
    print(f'{path.name}: no-cut sequences {clean_n}/{len(res)} = {clean_pct:.1%}')

    return res, summary, absent

In [ ]:
all_results = []
all_summaries = []
all_absent = []

for file in INPUT_FILES:
    res_i, summary_i, absent_i = scan_one_file(file)
    all_results.append(res_i)
    all_summaries.append(summary_i)
    all_absent.append(absent_i)

combined = pd.concat(all_results, ignore_index=True)
combined_summary = pd.concat(all_summaries, ignore_index=True)
combined_absent = pd.concat(all_absent, ignore_index=True)

combined.to_csv(OUTPUT_DIR / 'ALL_files_RE_scan.csv', index=False)
combined_summary.to_csv(OUTPUT_DIR / 'ALL_files_RE_summary.csv', index=False)
combined_absent.to_csv(OUTPUT_DIR / 'ALL_files_absent_RE_sites_by_file.csv', index=False)

# Keep these names for later cells / quick inspection.
df = all_results[0]
res = all_results[0]
summary = all_summaries[0]
enz_cols = list(RE_SITES.keys())

combined_summary

In [ ]:
# RE sites absent in every input file.
absent_sets = [
    set(summary_i.loc[summary_i['total_occurrences'] == 0, 'enzyme'])
    for summary_i in all_summaries
]

common_absent_enzymes = sorted(set.intersection(*absent_sets)) if absent_sets else []

common_absent_RE_sites = pd.DataFrame({
    'enzyme': common_absent_enzymes,
    'site': [RE_SITES[e] for e in common_absent_enzymes],
    'status': 'absent_in_all_files',
})

common_absent_RE_sites.to_csv(OUTPUT_DIR / 'common_absent_RE_sites.csv', index=False)
print(f'common absent RE sites: {len(common_absent_RE_sites)}')
print(f'saved -> {OUTPUT_DIR / "common_absent_RE_sites.csv"}')

common_absent_RE_sites

In [ ]:
# 查單條序列的詳細切位 (酶 -> 位置清單)
def site_detail(seq):
    seq = str(seq)
    return {enz: find_sites(seq, site) for enz, site in RE_SITES.items() if find_sites(seq, site)}

site_detail(df['Sequence'].iloc[0])

## Standardised report for the whole library

Uses the segment offsets recorded by 06 to say *where* each site sits.
A site inside BG5 / RE1 / RE2 / BG3 is by design; a site inside the promoter
or barcode, or spanning a junction, is the thing worth acting on.

Writes `outputs/07_re_scan_report.csv`.

In [ ]:
# === 07_re_scan_report.csv: per-candidate sites attributed to a segment ===
import pandas as pd

lib = pd.read_csv(require(RELEASE_OUT / "06_whole_sequence.csv", "assembled library"))
print("candidates:", len(lib))

RE1 = lib["re1_seq"].iloc[0]
RE2 = lib["re2_seq"].iloc[0]
print(f"construct RE1={RE1}  RE2={RE2}")


def segments(row):
    """(name, start, end) for one construct, end-exclusive."""
    return [
        ("BG5", 0, row["bg5_end"]),
        ("promoter", row["promoter_start"], row["promoter_end"]),
        ("RE1", row["re1_start"], row["re2_start"]),
        ("RE2", row["re2_start"], row["barcode_start"]),
        ("barcode", row["barcode_start"], row["bg3_start"]),
        ("BG3", row["bg3_start"], row["bg3_end"]),
    ]


def locate(pos, length, segs):
    """Which segment(s) a site at [pos, pos+length) touches."""
    return "+".join(n for n, a, b in segs if pos < b and pos + length > a)


# A site is "expected" only when it lies wholly inside one constant segment:
# the BsaI sites built into BG5/BG3, and RE1/RE2 themselves.
EXPECTED = {"BG5", "BG3", "RE1", "RE2"}

rows = []
for row in lib.itertuples(index=False):
    r = row._asdict()
    segs = segments(r)
    seq = r["full_sequence"]
    rec = {
        "candidate_id": r["candidate_id"],
        "source": r["source"],
        "full_length": r["full_length"],
    }
    n_expected = n_extra = 0
    extra_where = []
    for enz, site in RE_SITES.items():
        hits = find_re_sites(seq, site)
        extra = 0
        for p in hits:
            where = locate(p, len(site), segs)
            if where in EXPECTED:
                n_expected += 1
            else:
                extra += 1
                extra_where.append(f"{enz}@{p}({where})")
        rec[enz] = extra
        n_extra += extra
    rec["sites_expected"] = n_expected
    rec["sites_unexpected"] = n_extra
    rec["clean"] = n_extra == 0
    rec["unexpected_detail"] = "; ".join(extra_where)
    rows.append(rec)

report = pd.DataFrame(rows)
dest = RELEASE_OUT / "07_re_scan_report.csv"
report.to_csv(dest, index=False)
print(f"\nwrote {dest}  {report.shape}")


In [ ]:
# === Summary: what actually needs a decision ===
enz_cols = [e for e in RE_SITES if e in report.columns]

print(f"clean constructs: {int(report['clean'].sum())} / {len(report)} "
      f"({report['clean'].mean():.1%})\n")

print("clean rate by source:")
print(report.groupby("source")["clean"].agg(["size", "sum", "mean"]).to_string())

per_enzyme = (
    pd.DataFrame({
        "enzyme": enz_cols,
        "site": [RE_SITES[e] for e in enz_cols],
        "n_constructs": [(report[e] > 0).sum() for e in enz_cols],
        "n_sites": [report[e].sum() for e in enz_cols],
    })
    .sort_values("n_constructs", ascending=False)
    .reset_index(drop=True)
)
print("\nunexpected sites per enzyme (outside BG5/BG3/RE1/RE2):")
print(per_enzyme[per_enzyme["n_constructs"] > 0].to_string(index=False))

zero = per_enzyme[per_enzyme["n_constructs"] == 0]["enzyme"].tolist()
print(f"\nenzymes with zero unexpected sites library-wide: {zero}")
print("These are the safe candidates if RE1/RE2 need to be swapped.")

print("\nexample offenders:")
bad = report[~report["clean"]]
if len(bad):
    print(bad[["candidate_id", "source", "sites_unexpected", "unexpected_detail"]].head(10).to_string(index=False))
else:
    print("none")
